# TinySpring – Validation intelligente des traitements (Decision Tree)

Ce notebook entraîne un modèle `DecisionTreeClassifier` (scikit-learn) pour prédire si un traitement est :

- **ACCEPTÉ** (1)
- **REFUSÉ** (0)

Puis il exporte l'arbre en **JSON** au format attendu par le backend Spring Boot.

## Features (même ordre que le backend)
0. `has_ordonnance` (0/1)
1. `duration_days` (0..60)
2. `prises_per_day` (0..6)
3. `desc_len` (0..300)
4. `age_years` (0..12)
5. `has_chronic` (0/1)
6. `recent_incidents` (0..10)


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

np.random.seed(7)


## 1) Dataset d'exemple (simple)

Dans un projet réel, on entraînerait à partir d'un dataset exporté depuis la base (treatments déjà **VALIDE/REFUSE**).
Ici, on crée un dataset synthétique pour fournir un modèle fonctionnel et démontrable.

In [ ]:
N = 800

has_ordonnance = np.random.binomial(1, 0.92, N)
duration_days = np.clip(np.random.normal(7, 6, N).round(), 0, 60)
prises_per_day = np.clip(np.random.choice([0,1,2,3,4], size=N, p=[0.02,0.08,0.40,0.40,0.10]), 0, 6)
desc_len = np.clip(np.random.normal(80, 60, N).round(), 0, 300)
age_years = np.clip(np.random.normal(4, 2.2, N).round(), 0, 12)
has_chronic = np.random.binomial(1, 0.22, N)
recent_incidents = np.clip(np.random.poisson(1.3, N), 0, 10)

# Règles de label (syntétique) :
# - sans ordonnance => souvent refusé
# - 0 prise/jour => refusé
# - trop d'incidents + enfant très jeune + longue durée => plutôt à risque (on le met côté 0 dans ce dataset)
score = (
    2.2 * has_ordonnance
    + 0.6 * (prises_per_day >= 2)
    + 0.4 * (desc_len >= 20)
    - 0.6 * (recent_incidents >= 4)
    - 0.5 * ((age_years <= 2) & (duration_days >= 14))
    - 0.4 * ((duration_days >= 30) & (prises_per_day >= 4))
).astype(float)

y = (score >= 2.0).astype(int)  # 1=accepté, 0=refusé

df = pd.DataFrame({
    'has_ordonnance': has_ordonnance,
    'duration_days': duration_days,
    'prises_per_day': prises_per_day,
    'desc_len': desc_len,
    'age_years': age_years,
    'has_chronic': has_chronic,
    'recent_incidents': recent_incidents,
    'label_accept': y,
})

df.head()

## 2) Entraînement Decision Tree

In [ ]:
X = df[[
    'has_ordonnance',
    'duration_days',
    'prises_per_day',
    'desc_len',
    'age_years',
    'has_chronic',
    'recent_incidents'
]].values
y = df['label_accept'].values

clf = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=20,
    random_state=7
)
clf.fit(X, y)

train_acc = clf.score(X, y)
train_acc

## 3) Export JSON pour le backend

Le backend attend un arbre sous forme récursive avec :

- `leaf`: bool
- `featureIndex`: int
- `threshold`: float
- `count`: int
- `positives`: int
- `left` / `right`: node


In [ ]:
tree = clf.tree_

def build_node(node_id: int):
    # value shape: (1, n_classes)
    value = tree.value[node_id][0]
    count = int(tree.n_node_samples[node_id])
    positives = int(round(value[1])) if len(value) > 1 else 0
    feature = int(tree.feature[node_id])
    thr = float(tree.threshold[node_id])
    left_id = int(tree.children_left[node_id])
    right_id = int(tree.children_right[node_id])
    is_leaf = left_id == right_id

    node = {
        'leaf': bool(is_leaf),
        'featureIndex': int(feature) if not is_leaf else -1,
        'threshold': float(thr) if not is_leaf else 0.0,
        'count': count,
        'positives': positives,
        'left': None,
        'right': None,
    }

    if not is_leaf:
        node['left'] = build_node(left_id)
        node['right'] = build_node(right_id)

    return node

snapshot = {
    'sampleCount': int(len(df)),
    'root': build_node(0)
}

out_path = Path('..') / 'backend' / 'garderie' / 'models' / 'traitement_validation_tree.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding='utf-8')

str(out_path.resolve())